# Cell type annotation: three open questions

Read-only diagnostic. Re-uses the existing MapMyCells JSON and writes only to
`cell_type_annotation/_diagnostics/`.

| § | Question | Settled by |
|---|---|---|
| 1 | Are marker scores comparable across cell types, or does the per-gene z-score inflate rare types? | noise floor vs type abundance; false `Undefined` on clean clusters |
| 2 | How noisy is the cluster vote (enrichment argmax)? | purity of the winning type, vs plain majority |
| 3 | Does any of this change `cell_type_revised`, and what does letting QC-flagged cells vote do? | end-to-end comparison of the final column |
| 4 | Are `Undefined` cells mixed identity (segmentation errors) or just low quality? | counts and marker co-expression of `Undefined` vs assigned cells, raw and at matched counts |
| 5 | Do the labels look plausible at each step? | UMAP and spatial plots per labelling step, plus `main` |

Only helpers that are identical on `main` and `scalpel-annotation` come from the package,
so this runs on either checkout. The QC step (doubleMAD on the branch) runs as in the
script. On `main` it does not feed `cell_type_revised`, which is built from
`cell_type_mmc_raw`; the proposed design (§3) builds it from `cell_type_mmc_incl_low_quality`,
so flagged cells vote as `Undefined` and can then be relabelled by the vote and the revision.

In [ ]:
# ruff: noqa
import os, sys, json, logging, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import scipy.sparse as sp
from scipy.stats import spearmanr, mannwhitneyu

REPO = "/dss/dsshome1/0C/ra98gaq/Git/cellseg-benchmark-annot"   # <- your checkout
sys.path.insert(1, REPO)
import cellseg_benchmark.cell_annotation_utils as anno_utils
from cellseg_benchmark._constants import cell_type_colors

warnings.filterwarnings("ignore")
logger = logging.getLogger("diagnostics")
logger.setLevel(logging.INFO)
if not logger.handlers:
    h = logging.StreamHandler()
    h.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s]: %(message)s"))
    logger.addHandler(h)
pd.set_option("display.width", 160)


class Args:
    sample_name = "htra1_s3_r0"
    seg_method = "Proseg_3D_Cellpose_1_nuclei_model"
    data_dir = "/dss/dssfs03/pn52re/pn52re-dss-0001/cellseg-benchmark"
    mad_factor = 3.0
    leiden_res = 10.0


args = Args()
KEY = "cell_type_mmc_raw"          # the variant behind cell_type_revised on main
KEY_QC = "cell_type_mmc_incl_low_quality"   # proposed: QC-flagged cells vote as Undefined
HI, LO, DELTA = 0.5, 0.5, 0.25     # as passed by cell_type_annotation.py
MIN_SHARE = 0.10                   # floor tested in section 3

method_path = Path(args.data_dir, "samples", args.sample_name, "results", args.seg_method)
annotation_path = method_path / "cell_type_annotation"
out_path = annotation_path / "_diagnostics"
out_path.mkdir(parents=True, exist_ok=True)

## 0. Rebuild the pipeline state up to marker scoring

In [ ]:
from spatialdata import read_zarr

json_path = max(
    annotation_path.glob(f"mapmycells_out/*MapMyCells_{args.sample_name}_{args.seg_method}.json"),
    key=os.path.getmtime,
)
logger.info("Using %s", json_path.name)
with open(json_path, "rb") as src:
    mmc = anno_utils.process_mapmycells_output(json.load(src))

prefixes = ["allen", "allen_runner_up_1", "allen_runner_up_2"]
for lvl in [l for l in ["CLAS", "SUBC", "SUPT", "CLUS"] if f"allen_cor_{l}" in mmc]:
    for p in prefixes:
        anno_utils.mark_low_quality_mappings(mmc, target_column=p,
                                             mad_factor=args.mad_factor, level=lvl)
for p in prefixes:
    for s in ["SUBC", "SUBC_incl_low_quality"]:
        mmc[f"{p}_{s}"] = anno_utils.group_cell_types(mmc[f"{p}_{s}"])
mmc = mmc.merge(anno_utils.create_mixed_cell_types(df=mmc, diff_threshold=0.5),
                left_index=True, right_index=True, how="left")

adata = read_zarr(method_path / "sdata.zarr")["table"]
adata = adata[:, ~adata.var_names.str.startswith("Blank")]
adata.obsm["allen_cell_type_mapping"] = mmc.loc[adata.obs.index]
adata = anno_utils.process_adata(adata=adata, seg_method=args.seg_method, logger=logger)

leiden_col = f"leiden_res{args.leiden_res}".replace(".", "_")
if leiden_col not in adata.obs:
    sc.tl.leiden(adata, key_added=leiden_col, resolution=args.leiden_res)

marker_csv = Path(args.data_dir, "misc", "scRNAseq_ref_ABCAtlas_Yao2023Nature",
                  "marker_genes_df", "20250416_cell_type_markers_top50.csv")
marker_df = pd.read_csv(marker_csv)
marker_dict = {c: [g for g in marker_df[c].tolist() if pd.notna(g)]
               for c in marker_df.columns if c != "0"}
marker_dict.pop("Bergmann", None)

# As in the script: layer=None, and process_adata() has set X to the z-scored layer.
adata = anno_utils.score_cell_types(adata, marker_dict, top_n_genes=50, layer=None,
                                    score_prefix="score", logger=logger)
# Same markers on log-normalised counts, for comparison.
adata = anno_utils.score_cell_types(adata, marker_dict, top_n_genes=50,
                                    layer="volume_log1p_norm", score_prefix="scorelog",
                                    logger=logger)
print(adata.obs[leiden_col].nunique(), "clusters,", f"{len(adata):,} cells")

### Helpers (the two rules, rewritten over the cluster-mean matrix; same logic as the package)

In [ ]:
lc = leiden_col
cl = adata.obs[lc].astype(str)
n_cells = cl.value_counts()
ALIAS = {"Neurons-Dopa-Gaba": "Neurons-Dopa"}   # marker-table name -> MMC label

scored = [t for t in marker_dict if f"score_{t}" in adata.obs]
branch_map = {ALIAS.get(t, t): t for t in scored if ALIAS.get(t, t) in cell_type_colors}
legacy_map = {t: t for t in scored}
if "Neurons-Dopa-Gaba" in scored:
    legacy_map["Neurons-Dopa"] = "Neurons-Dopa-Gaba"


def cluster_labels(key, rule="enrichment", min_share=0.0, min_cells=100):
    # enrichment at min_share=0 is rank-identical to main's column-normalised crosstab
    counts = pd.crosstab(cl, adata.obs[key])
    counts = counts.loc[:, counts.sum(axis=0) >= min_cells]
    within = counts.div(counts.sum(axis=1), axis=0)
    lab = within.idxmax(axis=1)
    if rule == "enrichment":
        enr = within.div(counts.sum(axis=0) / counts.to_numpy().sum(), axis=1)
        enr = enr.where(within >= min_share)
        ok = enr.notna().any(axis=1)
        lab[ok] = enr[ok].idxmax(axis=1)
    return lab, within


def cluster_means(prefix, mapping):
    cols = [f"{prefix}_{v}" for v in mapping.values()]
    m = adata.obs.groupby(cl, observed=True)[sorted(set(cols))].mean()[cols]
    m.columns = list(mapping.keys())
    return m


def revise(labels, means, legacy=False, hi=HI, lo=LO, delta=DELTA):
    # legacy=True: main's guard (skipped when the label has no score of its own)
    final = {}
    for c, lab in labels.items():
        s = means.loc[c].dropna().sort_values(ascending=False)
        if s.empty:
            final[c] = "Undefined"
            continue
        top, top_s = s.index[0], s.iloc[0]
        own = s.get(lab, np.nan)
        ref = own if (legacy or pd.notna(own)) else (s.iloc[1] if len(s) > 1 else -np.inf)
        if top_s >= hi:
            d = top_s - ref
            final[c] = top if (np.isnan(d) or d > delta) else lab
        elif (s < lo).all():
            final[c] = "Undefined"
        else:
            final[c] = lab
    final = pd.Series(final)
    lab = pd.Series(labels)[final.index]
    decision = pd.Series(np.where(final == "Undefined", "undefined",
                                  np.where(final == lab, "kept", "reassigned")),
                         index=final.index)
    return final, decision


def to_cells(per_cluster):
    x = cl.map(per_cluster).astype(object)
    return x.where(x.isin(list(cell_type_colors)), "NaN (-> Low-Read-Cells)")


def w(clusters):
    # cells in a set of clusters
    return int(n_cells.reindex(list(clusters)).fillna(0).sum())


means_z = cluster_means("score", branch_map)
means_log = cluster_means("scorelog", branch_map)
maj_lab, within_maj = cluster_labels(KEY, rule="majority")
purity = within_maj.max(axis=1)
within_raw = pd.crosstab(cl, adata.obs[KEY], normalize="index")   # all types, no min_cells
sample_share = adata.obs[KEY].value_counts(normalize=True)

## 1. Marker score scale

**Hypothesis.** Scores are computed on the per-gene z-scored layer. A marker expressed in a
fraction *p* of cells reaches z ≈ √((1−p)/p) in those cells: about 1 at p = 0.5, about 10 at
p = 0.01. So rare-type scores sit on a larger scale, and one fixed cutoff (0.5) and one
argmax across types are not comparable.

### 1a. What the score actually is

In [ ]:
Z = adata.layers["zscore"]
Z = Z.toarray() if sp.issparse(Z) else np.asarray(Z)
assert np.allclose(np.asarray(adata.X[:50]), Z[:50]), "X is not the zscore layer"
print(f"zscore layer: per-gene |mean| max {np.abs(Z.mean(0)).max():.2e}, "
      f"sd range {Z.std(0).min():.2f}-{Z.std(0).max():.2f}")
print("score_genes picks control genes by binning mean expression. With every gene mean at ~0")
print("that binning is arbitrary, so the score should reduce to the mean z of the markers.\n")

C = adata.layers["counts"]
C = C.toarray() if sp.issparse(C) else np.asarray(C)
var_idx = pd.Series(np.arange(adata.n_vars), index=adata.var_names)

rows = []
for t in scored:
    genes = [g for g in marker_dict[t][:50] if g in var_idx.index]
    idx = var_idx[genes].to_numpy()
    p = (C[:, idx] > 0).mean(0)
    expr = C[:, idx] > 0
    z_expr = np.nanmedian(np.where(expr, Z[:, idx], np.nan), axis=0)
    label = ALIAS.get(t, t)
    rows.append({
        "type": label, "n_genes": len(genes),
        "sample_share": sample_share.get(label, 0.0),
        "marker_prevalence": np.median(p),
        "z_in_expressing_cells": np.nanmedian(z_expr),
        "z_theory": np.median(np.sqrt((1 - p) / np.clip(p, 1e-9, None))),
        "r(score, mean marker z)": np.corrcoef(Z[:, idx].mean(1), adata.obs[f"score_{t}"])[0, 1],
    })
mech = pd.DataFrame(rows).set_index("type").sort_values("sample_share")
mech.round(3)

Read-out: `r(score, mean marker z)` near 1 means the score is simply the mean z of the
markers. If `z_in_expressing_cells` rises as `marker_prevalence` falls (and tracks
`z_theory`), the scale mechanism is real.

### 1b. Noise floor and signal per type

For each type, clusters where it is the plain majority (≥ 50% of MMC labels) are
**positive**; clusters where it holds < 5% are **negative**. A usable score has positives
above the cutoff and negatives below it.

In [ ]:
def pos_neg(means):
    rows = []
    for t in means.columns:
        share = within_raw[t] if t in within_raw else pd.Series(0.0, index=means.index)
        pos = means.loc[share.index[share >= 0.5], t].dropna()
        neg = means.loc[share.index[share < 0.05], t].dropna()
        r = {"type": t, "sample_share": sample_share.get(t, 0.0),
             "n_genes": mech["n_genes"].get(t, np.nan),
             "n_pos": len(pos), "n_neg": len(neg),
             "pos_q05": pos.quantile(0.05) if len(pos) else np.nan,
             "pos_median": pos.median() if len(pos) else np.nan,
             "neg_median": neg.median() if len(neg) else np.nan,
             "neg_q95": neg.quantile(0.95) if len(neg) else np.nan,
             "neg_max": neg.max() if len(neg) else np.nan}
        r["auc"] = (mannwhitneyu(pos, neg).statistic / (len(pos) * len(neg))
                    if len(pos) and len(neg) else np.nan)
        rows.append(r)
    return pd.DataFrame(rows).set_index("type").sort_values("sample_share")


pn_z, pn_log = pos_neg(means_z), pos_neg(means_log)

for name, pn in [("z-scored (as in script)", pn_z), ("log-normalised", pn_log)]:
    ok = pn[["sample_share", "neg_q95"]].dropna()
    rho = spearmanr(np.log10(ok.sample_share.clip(lower=1e-5)), ok.neg_q95)[0]
    print(f"{name:<24} Spearman(log sample share, noise floor neg_q95) = {rho:+.2f}")

print(f"\nz-scored, cutoff {HI}:")
print("  types whose noise floor reaches the cutoff (neg_q95 >= 0.5):",
      pn_z.index[pn_z.neg_q95 >= HI].tolist())
print("  types whose real clusters fall below it (pos_q05 < 0.5):   ",
      pn_z.index[pn_z.pos_q05 < HI].tolist())
pn_z.round(3)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)
order = pn_z.index.tolist()
for ax, means, title in [(axes[0], means_z, "z-scored layer (as in script)"),
                         (axes[1], means_log, "log-normalised layer")]:
    for i, t in enumerate(order):
        share = within_raw[t] if t in within_raw else pd.Series(0.0, index=means.index)
        neg = means.loc[share.index[share < 0.05], t]
        pos = means.loc[share.index[share >= 0.5], t]
        ax.scatter(np.full(len(neg), i) + np.random.uniform(-.25, .25, len(neg)), neg,
                   s=5, c="lightgrey", label="negative clusters" if i == 0 else None)
        ax.scatter(np.full(len(pos), i) + np.random.uniform(-.15, .15, len(pos)), pos,
                   s=14, c="C3", label="positive clusters" if i == 0 else None)
    if means is means_z:
        ax.axhline(HI, color="k", ls="--", lw=1, label=f"cutoff {HI}")
    ax.set_ylabel("cluster-mean score")
    ax.set_title(title)
    ax.legend(fontsize=8, loc="upper left")
axes[1].set_xticks(range(len(order)))
axes[1].set_xticklabels([f"{t} ({100 * pn_z.sample_share[t]:.1f}%)" for t in order],
                        rotation=90, fontsize=7)
axes[1].set_xlabel("cell type (share of sample), rare -> common")
plt.tight_layout()
plt.savefig(out_path / "1_score_scale.png", dpi=150)
plt.show()

### 1c. Does the scale do damage on clusters MMC is confident about?

Confident clusters: plain-majority purity ≥ 0.5. MMC is not ground truth, but on these
clusters a sound marker score should mostly agree with it, and should rarely call them
`Undefined`.

In [ ]:
conf = purity.index[purity >= 0.5]
std_z = (means_z - means_z.mean()) / means_z.std()   # candidate fix: per type across clusters

rows = []
for name, m in [("z-scored (as in script)", means_z), ("log-normalised", means_log),
                ("z-scored, standardised per type", std_z)]:
    top = m.loc[conf].idxmax(axis=1)
    agree = top == maj_lab[conf]
    rows.append({"scale": name, "confident clusters": len(conf),
                 "cells": w(conf), "argmax agrees with MMC (% cells)":
                 100 * w(agree.index[agree]) / w(conf)})
agree_tbl = pd.DataFrame(rows).set_index("scale").round(1)
display(agree_tbl)

final_c, dec_c = revise(maj_lab, means_z)
d = dec_c[conf]
print(f"\nCurrent rule (cutoff {HI}) on confident clusters:")
for k in ["kept", "reassigned", "undefined"]:
    print(f"  {k:<11} {100 * w(d.index[d == k]) / w(conf):5.1f}% of cells")

top_z = means_z.loc[conf].idxmax(axis=1)
wrong = pd.DataFrame({"mmc_majority": maj_lab[conf], "marker_argmax": top_z,
                      "cells": n_cells[conf]})
wrong = wrong[wrong.mmc_majority != wrong.marker_argmax]
wrong_by_type = (wrong.groupby("marker_argmax")["cells"].sum().rename("cells_won_against_mmc")
                 .to_frame().join(pn_z[["sample_share", "n_genes"]])
                 .sort_values("cells_won_against_mmc", ascending=False))
print("\nOn confident clusters, which types win the z-scored argmax against MMC:")
wrong_by_type.round(4)

**Verdict for §1.** The scale problem is confirmed if all three hold:
1. the Spearman correlation is clearly negative for the z-scored layer, but weaker for the log layer;
2. `cells_won_against_mmc` is dominated by rare types;
3. standardising per type raises agreement with MMC.

The cutoff itself is too strict if a large share of confident-cluster cells come out
`undefined`. If the log layer or standardised scores do not do better, the scale is not
the problem, and the 17.5% `Undefined` needs another explanation.

## 2. The cluster vote

In [ ]:
enr0, within = cluster_labels(KEY, "enrichment", 0.0)
enr_f, _ = cluster_labels(KEY, "enrichment", MIN_SHARE)
won = pd.Series({c: within.loc[c, t] for c, t in enr0.items()})
diff = enr0.index[enr0 != maj_lab]

print(f"clusters where the enrichment winner is not the majority: {len(diff)} / {len(enr0)} "
      f"({w(diff):,} cells, {100 * w(diff) / len(adata):.1f}%)")
print(f"  of those, winner holds < 10% of its cluster: {(won[diff] < 0.10).sum()}")
print(f"  min_share={MIN_SHARE} still differs from majority in "
      f"{(enr_f != maj_lab).sum()} clusters")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(won, bins=40, alpha=.7, label="enrichment winner (main)")
ax.hist(purity, bins=40, alpha=.7, label="plain majority")
ax.axvline(MIN_SHARE, color="k", ls="--", lw=1, label=f"min_share {MIN_SHARE}")
ax.set_xlabel("share of the cluster held by the winning type")
ax.set_ylabel("clusters")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(out_path / "2_cluster_purity.png", dpi=150)
plt.show()

If the enrichment winners pile up below 0.1 while the majority sits well above, the vote
is picking noise. Whether that matters is §3.

## 3. End to end: `cell_type_revised`

All arms use the script's thresholds. The first four differ only in the cluster rule and
the guard, on `cell_type_mmc_raw` as on `main`. The last two are the proposed design:
labels from `cell_type_mmc_incl_low_quality`, so QC-flagged cells vote as `Undefined`.

In [ ]:
arms = {
    "main":                      (enr0, cluster_means("score", legacy_map), True),
    "branch":                    (enr0, means_z, False),
    f"branch min_share={MIN_SHARE}": (enr_f, means_z, False),
    "branch plain majority":     (maj_lab, means_z, False),
    "QC votes, min_share=0":     (cluster_labels(KEY_QC, "enrichment", 0.0)[0], means_z, False),
    f"QC votes, min_share={MIN_SHARE}": (cluster_labels(KEY_QC, "enrichment", MIN_SHARE)[0], means_z, False),
}
final_cells, cluster_cells, decisions = {}, {}, {}
for name, (labels, means, legacy) in arms.items():
    f, dec = revise(labels, means, legacy=legacy)
    final_cells[name] = to_cells(f)
    cluster_cells[name] = to_cells(labels)
    decisions[name] = cl.map(dec)

ref_final, ref_cluster = final_cells["main"], cluster_cells["main"]
rows = []
for name in arms:
    cl_diff = cluster_cells[name] != ref_cluster
    fin_diff = final_cells[name] != ref_final
    rows.append({
        "arm": name,
        "% Undefined": 100 * (final_cells[name] == "Undefined").mean(),
        "% kept": 100 * (decisions[name] == "kept").mean(),
        "% reassigned": 100 * (decisions[name] == "reassigned").mean(),
        "% cluster label != main": 100 * cl_diff.mean(),
        "% final label != main": 100 * fin_diff.mean(),
        "% of cluster changes that survive revision":
            100 * (cl_diff & fin_diff).sum() / cl_diff.sum() if cl_diff.any() else np.nan,
    })
summary3 = pd.DataFrame(rows).set_index("arm").round(2)
summary3

In [ ]:
# Is the final label set by the cluster vote or by the marker argmax?
top_all = means_z.idxmax(axis=1)
f_main = final_cells["main"]
src = np.select([f_main == "Undefined", f_main == cl.map(top_all)],
                ["Undefined (score rule)", "= marker argmax"], "= cluster vote only")
print("main, where cell_type_revised comes from (% cells):")
print((100 * pd.Series(src).value_counts(normalize=True)).round(1).to_string(), "\n")

per_type = pd.DataFrame({n: s.value_counts() for n, s in final_cells.items()}).fillna(0).astype(int)
per_type.index.name = "cell type"
per_type["main, no revision"] = cluster_cells["main"].value_counts().reindex(per_type.index).fillna(0).astype(int)
per_type.sort_values("main", ascending=False)

**Verdict for §3.**

- If `= cluster vote only` is small, the marker score decides the final label, and the
  scale fix from §1 is where the leverage is.
- If `% final label != main` is small for the majority and `min_share` arms, the noisy
  vote from §2 is mostly overwritten by the revision. Switch to plain majority or
  `min_share` for correctness and clarity, but expect little change in DEA results.
- If it is large, pick the cluster rule before any rerun.
- `% Undefined` identical across arms means the 17.5% comes from the score cutoff alone.
- The `QC votes` arms show what the switch to the filtered labels does. A small
  difference from the matching `branch` arm is expected: a flagged cell only changes its
  final label if it flips its cluster's vote.

## 4. What are the `Undefined` cells?

The final share of `Undefined` is meant as a segmentation QC, which assumes those cells
are **mixed identity** (merged or contaminated cells), not just **low quality** (too few
transcripts to map). Both look the same to the rules: low correlation in MMC, no marker
score above the cutoff. This section tells them apart per cell, using the proposed arm.

- low quality: fewer counts and genes than assigned cells, no extra marker co-expression
- mixed identity: normal counts, a second cell type's markers clearly up, a small gap
  between the MMC winner and runner-up

In [ ]:
ARM = f"QC votes, min_share={MIN_SHARE}"
final = final_cells[ARM]
qc_flag = adata.obs[KEY_QC].astype(str).eq("Undefined")
group = pd.Series(np.where(final == "Undefined", "Undefined (final)",
                  np.where(qc_flag, "QC-flagged, relabelled", "assigned")), index=adata.obs.index)

S = adata.obs[[f"score_{v}" for v in branch_map.values()]].to_numpy()
S_sorted = np.sort(S, axis=1)[:, ::-1]
per_cell = pd.DataFrame({
    "group": group,
    "total_counts": np.asarray(C.sum(1)).ravel(),
    "n_genes": np.asarray((C > 0).sum(1)).ravel(),
    "top1_score": S_sorted[:, 0],
    "top2_score": S_sorted[:, 1],
    "n_types_above_cutoff": (S >= HI).sum(1),
    "mmc_prob_gap": pd.to_numeric(adata.obs["cell_type_mmc_rup1_diff_prob"], errors="coerce"),
    "mmc_is_mixed": adata.obs["cell_type_mmc_is_mixed"].astype(str).eq("mixed"),
}, index=adata.obs.index)
vol = next((c for c in ["volume", "area"] if c in adata.obs), None)
if vol:
    per_cell[vol] = pd.to_numeric(adata.obs[vol], errors="coerce")

summary4 = per_cell.groupby("group").agg(
    n=("group", "size"),
    **{f"median_{c}": (c, "median") for c in per_cell.columns
       if c not in ("group", "mmc_is_mixed", "n_types_above_cutoff")},
    pct_2plus_types_above_cutoff=("n_types_above_cutoff", lambda x: 100 * (x >= 2).mean()),
    pct_mmc_mixed=("mmc_is_mixed", lambda x: 100 * x.mean()),
).T.round(3)
display(summary4)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for ax, col in zip(axes, ["total_counts", "top2_score", "mmc_prob_gap"]):
    for g, sub in per_cell.groupby("group"):
        ax.hist(sub[col].dropna(), bins=60, density=True, histtype="step", lw=1.5, label=g)
    ax.set_xlabel(col)
axes[0].set_xscale("log")
axes[0].legend(fontsize=7)
plt.tight_layout()
plt.savefig(out_path / "4_undefined_identity.png", dpi=150)
plt.show()

**Verdict for §4.** If `Undefined (final)` has clearly lower counts and genes but no
higher `top2_score`, `pct_2plus_types_above_cutoff` or `pct_mmc_mixed` than `assigned`,
these are low-quality cells. Their share then reflects detection sensitivity (small or
nuclei-only masks score badly), not segmentation errors. The reverse pattern supports
the metric.

Separately, both thresholds are relative to the sample: MAD per MMC group, and marker
scores on data z-scored within the sample. A method that is uniformly worse shifts the
reference with it, so compare `% Undefined` across segmentation methods with that in
mind, or fix the cutoffs from one reference method.

### 4b. The same comparison at matched counts

Fewer counts alone make MapMyCells less certain (smaller gap, more `mixed`) and lower
every marker score. So the question is whether `Undefined` cells still look more mixed
than assigned cells **with the same number of counts**. Cells are binned into count
deciles; each group is compared with assigned cells inside the same bin, and the
differences are averaged over the bins, weighted by that group's cells.

In [ ]:
MIN_N = 30   # cells per group and bin needed for a comparison
edges = np.unique(np.quantile(per_cell.total_counts, np.linspace(0, 1, 11)))
per_cell["count_bin"] = pd.cut(per_cell.total_counts, edges, include_lowest=True)

metrics = {
    "pct_mmc_mixed": ("mmc_is_mixed", lambda x: 100 * x.mean()),
    "median_mmc_prob_gap": ("mmc_prob_gap", "median"),
    "median_top2_score": ("top2_score", "median"),
    "pct_2plus_types": ("n_types_above_cutoff", lambda x: 100 * (x >= 2).mean()),
}
strat = per_cell.groupby(["count_bin", "group"], observed=True).agg(
    n=("group", "size"), median_counts=("total_counts", "median"), **metrics)
overall = per_cell.groupby("group").agg(**metrics)
wide = strat.unstack("group")

rows = []
for g in ["Undefined (final)", "QC-flagged, relabelled"]:
    if g not in overall.index:
        continue
    ok = (wide[("n", g)] >= MIN_N) & (wide[("n", "assigned")] >= MIN_N)
    wts = wide.loc[ok, ("n", g)]
    r = {"group": g,
         "% of group in comparable bins": 100 * wts.sum() / (per_cell.group == g).sum()}
    for m in metrics:
        r[f"{m}: raw diff"] = overall.loc[g, m] - overall.loc["assigned", m]
        diff = wide.loc[ok, (m, g)] - wide.loc[ok, (m, "assigned")]
        r[f"{m}: count-matched diff"] = np.average(diff, weights=wts) if ok.any() else np.nan
    rows.append(r)
matched = pd.DataFrame(rows).set_index("group").T.round(3)
display(matched)

fig, axes = plt.subplots(1, len(metrics), figsize=(16, 3.6))
for ax, m in zip(axes, metrics):
    for g in strat.index.get_level_values("group").unique():
        sub = strat.xs(g, level="group")
        sub = sub[sub.n >= MIN_N]
        ax.plot(sub.median_counts, sub[m], "o-", label=g)
    ax.set_xscale("log")
    ax.set_xlabel("median counts in bin")
    ax.set_title(m, fontsize=9)
axes[0].legend(fontsize=7)
plt.tight_layout()
plt.savefig(out_path / "4b_count_matched.png", dpi=150)
plt.show()
strat.round(3)

**Verdict for §4b.** If the count-matched differences shrink towards 0 compared with the
raw ones, the mixed-looking signal was a count effect and `Undefined` means low quality.
If `pct_mmc_mixed` stays clearly higher (or `median_mmc_prob_gap` clearly lower) at
matched counts, there is ambiguity beyond counts, which is consistent with mixed identity.
Check `% of group in comparable bins` first: if it is low, most `Undefined` cells have
fewer counts than almost any assigned cell and cannot be matched at all, which is itself
evidence for low quality.

## 5. Labels at each step, for a visual check

One row per step of the proposed pipeline, then `main`'s final label for comparison.
Left: UMAP, right: tissue. Look for rare types (Neurons-Dopa, Neurons-Other,
Astroependymal, BAMs) appearing only after the marker revision and covering large
contiguous regions, and for `Undefined` forming its own low-count island rather than
sitting between cell types.

In [ ]:
steps = {
    "1. MMC per cell": adata.obs[KEY].astype(str),
    "2. after doubleMAD QC (flagged = Undefined)": adata.obs[KEY_QC].astype(str),
    f"3. cluster vote ({ARM})": cluster_cells[ARM],
    f"4. after marker revision ({ARM})": final_cells[ARM],
    "main: cell_type_revised": final_cells["main"],
}
present = set().union(*[set(v.unique()) for v in steps.values()])
cats = [c for c in cell_type_colors if c in present]
extra = sorted(present - set(cats))
palette = {**{c: cell_type_colors[c] for c in cats}, **{c: "#555555" for c in extra}}

bases = ["umap"] + (["spatial"] if "spatial" in adata.obsm else [])
fig, axes = plt.subplots(len(steps), len(bases), figsize=(9 * len(bases), 7.5 * len(steps)),
                         squeeze=False)
for r, (title, labels) in enumerate(steps.items()):
    adata.obs["_step"] = pd.Categorical(labels, categories=cats + extra)
    for c, basis in enumerate(bases):
        sc.pl.embedding(adata, basis=basis, color="_step", palette=palette,
                        size=(220000 if basis == "umap" else 110000) / adata.n_obs,
                        legend_loc="on data" if basis == "umap" else None,
                        legend_fontsize=7, legend_fontoutline=1.5,
                        title=f"{title} ({basis})", ax=axes[r, c], show=False)
        axes[r, c].set_aspect("equal")
adata.obs.drop(columns="_step", inplace=True)
handles = [plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=palette[k],
                      markersize=8) for k in cats + extra]
fig.legend(handles, cats + extra, loc="center right", fontsize=9, frameon=False)
plt.subplots_adjust(right=0.85, wspace=0.05, hspace=0.12)
plt.savefig(out_path / "5_labels_by_step.png", dpi=110, bbox_inches="tight")
plt.show()

counts_by_step = pd.DataFrame({k: v.value_counts() for k, v in steps.items()}).fillna(0).astype(int)
counts_by_step.sort_values(counts_by_step.columns[0], ascending=False)

In [ ]:
summary4.to_csv(out_path / "4_undefined_identity.csv")
strat.to_csv(out_path / "4b_count_bins.csv")
matched.to_csv(out_path / "4b_count_matched.csv")
counts_by_step.to_csv(out_path / "5_counts_by_step.csv")
mech.to_csv(out_path / "1a_score_mechanism.csv")
pn_z.to_csv(out_path / "1b_pos_neg_zscore.csv")
pn_log.to_csv(out_path / "1b_pos_neg_log.csv")
agree_tbl.to_csv(out_path / "1c_agreement.csv")
wrong_by_type.to_csv(out_path / "1c_wins_against_mmc.csv")
summary3.to_csv(out_path / "3_end_to_end.csv")
per_type.to_csv(out_path / "3_per_type_counts.csv")
print("written to", out_path)